# 🔬 SurgiVision — EndoViT+DPT Segmentation Training
**CholecSeg8k · Few-shot (4 videos) · T4 GPU · ~2-3hrs**

### Steps:
1. Connect to GPU runtime: `Runtime → Change runtime type → T4 GPU`
2. Run cells top to bottom
3. At Cell 3, upload your zip from Mac
4. Come back when Cell 7 finishes — download the checkpoint

---

In [ ]:
# ─── CELL 1: Check GPU ───────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — change runtime to T4!"}')

In [ ]:
# ─── CELL 2: Install dependencies ────────────────────────────────
!pip install -q timm==0.9.16 einops==0.6.1 segmentation-models-pytorch==0.3.3
print('✓ Dependencies installed')

In [ ]:
# ─── CELL 3: Upload dataset from your Mac ────────────────────────
# On your Mac, first create the zip:
#   cd '/Users/matteomeister/Documents/Medical Devices/Projects/EndoViT'
#   zip -r ~/Desktop/CholecSeg8k_preprocessed.zip datasets/CholecSeg8k/data_preprocessed/
# Then upload it here:

from google.colab import files
import os

print('Upload CholecSeg8k_preprocessed.zip from your Mac...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

!mkdir -p /content/data
!unzip -q "{zip_name}" -d /content/data/

# Find where images actually are
import glob
sample = glob.glob('/content/data/**/*_endo.png', recursive=True)[:3]
print(f'Sample frames found: {sample}')
print(f'Total frames: {len(glob.glob("/content/data/**/*_endo.png", recursive=True))}')

In [ ]:
# ─── CELL 4: Download EndoViT backbone checkpoint ─────────────────
# Download from Google Drive (the endovit_seg.pth you already have)
# We'll use gdown to grab it directly

!pip install -q gdown
import gdown
import os

os.makedirs('/content/checkpoints', exist_ok=True)

print('Downloading EndoViT_Seg backbone checkpoint (~330MB)...')
gdown.download(
    'https://drive.google.com/uc?id=1NJ-4ZL40kHA_WZ1NylahaS84FcvnigjF',
    '/content/checkpoints/endovit_seg.pth',
    quiet=False
)

import torch
ckpt = torch.load('/content/checkpoints/endovit_seg.pth', map_location='cpu')
print(f'✓ Checkpoint loaded. Keys: {list(ckpt.keys())}')
print(f'  Epoch: {ckpt.get("epoch", "N/A")}')

In [ ]:
# ─── CELL 5: Build dataset ────────────────────────────────────────
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
import glob
import os
import re

# ── CholecSeg8k 13-class color map ──
# Maps RGB color → class index
COLOR_TO_CLASS = {
    (0,   0,   0):   0,   # Background
    (80,  80,  80):  1,   # Abdominal Wall
    (160, 0,   0):   2,   # Liver
    (0,   0,   160): 3,   # Gastrointestinal Tract
    (160, 160, 0):   4,   # Fat
    (0,   160, 160): 5,   # Grasper
    (160, 0,   160): 6,   # Connective Tissue
    (80,  160, 0):   7,   # Blood
    (160, 80,  0):   8,   # Cystic Duct
    (0,   80,  160): 9,   # L-Hook Electrocautery
    (0,   160, 80):  10,  # Gallbladder
    (80,  0,   160): 11,  # Hepatic Vein
    (160, 160, 160): 12,  # Liver Ligament
}

def color_mask_to_class_mask(color_mask_path):
    """Convert RGB color mask to single-channel class index mask."""
    img = np.array(Image.open(color_mask_path).convert('RGB'))
    h, w = img.shape[:2]
    class_mask = np.zeros((h, w), dtype=np.int64)
    for color, cls_idx in COLOR_TO_CLASS.items():
        match = np.all(img == np.array(color), axis=2)
        class_mask[match] = cls_idx
    return class_mask

class CholecSeg8kDataset(Dataset):
    def __init__(self, data_root, video_ids, image_size=224, augment=True):
        self.image_size = image_size
        self.augment    = augment
        self.samples    = []

        # Find all frames for given video IDs
        for vid_id in video_ids:
            pattern = f'{data_root}/**/video{vid_id:02d}/**/*_endo.png'
            frames  = glob.glob(pattern, recursive=True)
            # Also try without nested structure
            if not frames:
                pattern = f'{data_root}/**/*_endo.png'
                all_frames = glob.glob(pattern, recursive=True)
                frames = [f for f in all_frames
                         if f'video{vid_id:02d}' in f or f'video{vid_id}' in f]

            for frame_path in frames:
                # Find corresponding color mask
                mask_path = frame_path.replace('_endo.png', '_endo_color_mask.png')
                if os.path.exists(mask_path):
                    self.samples.append((frame_path, mask_path))

        print(f'Dataset: {len(self.samples)} samples from videos {video_ids}')

        # EndoViT normalization
        self.img_transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.3464, 0.2280, 0.2228],
                                  std=[0.2520, 0.2128, 0.2093])
        ])
        self.mask_resize = transforms.Resize(
            (image_size, image_size),
            interpolation=transforms.InterpolationMode.NEAREST
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]

        img  = Image.open(img_path).convert('RGB')
        img_tensor = self.img_transform(img)

        class_mask = color_mask_to_class_mask(mask_path)
        mask_pil   = Image.fromarray(class_mask.astype(np.uint8))
        mask_pil   = self.mask_resize(mask_pil)
        mask_tensor = torch.from_numpy(np.array(mask_pil)).long()

        return img_tensor, mask_tensor

# ── Find data root ──
all_pngs = glob.glob('/content/data/**/*_endo.png', recursive=True)
if all_pngs:
    # Find common root
    DATA_ROOT = '/content/data'
    print(f'Found {len(all_pngs)} total frames in {DATA_ROOT}')

    # Show available video IDs
    vid_ids_found = set()
    for p in all_pngs:
        m = re.search(r'video(\d+)', p)
        if m: vid_ids_found.add(int(m.group(1)))
    print(f'Available video IDs: {sorted(vid_ids_found)}')
else:
    print('ERROR: No frames found. Check upload path.')

In [ ]:
# ─── CELL 6: Build model ──────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial
from timm.models.vision_transformer import VisionTransformer

NUM_CLASSES = 13
IMAGE_SIZE  = 224
PATCH_SIZE  = 16
EMBED_DIM   = 768
NUM_PATCHES = (IMAGE_SIZE // PATCH_SIZE) ** 2  # 196
PATCH_HW    = IMAGE_SIZE // PATCH_SIZE          # 14

class DPTReassemble(nn.Module):
    """Reassembles ViT token output into spatial feature map at different scales."""
    def __init__(self, embed_dim=768, out_channels=256, scale_factor=1):
        super().__init__()
        self.proj = nn.Conv2d(embed_dim, out_channels, kernel_size=1)
        self.scale = scale_factor
        if scale_factor > 1:
            self.up = nn.ConvTranspose2d(out_channels, out_channels,
                                          kernel_size=scale_factor, stride=scale_factor)
        elif scale_factor < 1:
            k = int(1 / scale_factor)
            self.up = nn.Conv2d(out_channels, out_channels, kernel_size=k, stride=k)
        else:
            self.up = nn.Identity()

    def forward(self, x):
        # x: [B, N+1, C] — remove CLS, reshape
        B, _, C = x.shape
        x = x[:, 1:]  # remove CLS token
        x = x.permute(0, 2, 1).reshape(B, C, PATCH_HW, PATCH_HW)
        x = self.proj(x)
        x = self.up(x)
        return x


class DPTFusionBlock(nn.Module):
    def __init__(self, channels=256):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x, skip=None):
        if skip is not None:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = x + skip
        return self.conv(x)


class EndoViTDPT(nn.Module):
    """EndoViT backbone + DPT segmentation head."""
    def __init__(self, num_classes=13):
        super().__init__()

        # ── ViT backbone ──
        self.backbone = VisionTransformer(
            patch_size=PATCH_SIZE, embed_dim=EMBED_DIM,
            depth=12, num_heads=12, mlp_ratio=4, qkv_bias=True,
            norm_layer=partial(nn.LayerNorm, eps=1e-6)
        )

        # Hook intermediate layers 3, 6, 9, 11
        self._feats = {}
        for layer_idx in [2, 5, 8, 11]:
            self.backbone.blocks[layer_idx].register_forward_hook(
                lambda m, inp, out, i=layer_idx: self._feats.update({i: out})
            )

        # ── DPT Reassemble — 4 scales ──
        self.reassemble = nn.ModuleList([
            DPTReassemble(EMBED_DIM, 256, scale_factor=4),  # layer 3 → 56×56
            DPTReassemble(EMBED_DIM, 256, scale_factor=2),  # layer 6 → 28×28
            DPTReassemble(EMBED_DIM, 256, scale_factor=1),  # layer 9 → 14×14
            DPTReassemble(EMBED_DIM, 256, scale_factor=0.5),# layer 11→  7×7
        ])

        # ── DPT Fusion — bottom-up ──
        self.fusion = nn.ModuleList([
            DPTFusionBlock(256),
            DPTFusionBlock(256),
            DPTFusionBlock(256),
            DPTFusionBlock(256),
        ])

        # ── Segmentation head ──
        self.head = nn.Sequential(
            nn.Conv2d(256, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
            nn.Conv2d(128, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Upsample(size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=True),
            nn.Conv2d(64, num_classes, 1),
        )

    def forward(self, x):
        self._feats.clear()
        _ = self.backbone(x)
        layer_feats = [self._feats[i] for i in [2, 5, 8, 11]]

        # Reassemble to spatial maps
        maps = [self.reassemble[i](layer_feats[i]) for i in range(4)]

        # Fuse bottom-up: start from smallest, work up
        x = self.fusion[3](maps[3])
        x = self.fusion[2](x, maps[2])
        x = self.fusion[1](x, maps[1])
        x = self.fusion[0](x, maps[0])

        return self.head(x)


def load_endovit_backbone(model, ckpt_path):
    """Load EndoViT MAE weights into ViT backbone, skip decoder keys."""
    ckpt = torch.load(ckpt_path, map_location='cpu')
    state = ckpt.get('model', ckpt)

    # Filter: keep only encoder keys (skip decoder_*, mask_token, decoder_pos_embed)
    encoder_keys = {
        k: v for k, v in state.items()
        if not any(skip in k for skip in ['decoder', 'mask_token'])
    }

    missing, unexpected = model.backbone.load_state_dict(encoder_keys, strict=False)
    print(f'Backbone loaded ✓ — missing: {len(missing)}, unexpected: {len(unexpected)}')
    if missing[:3]: print(f'  First missing: {missing[:3]}')
    return model


# Build and load
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

model = EndoViTDPT(num_classes=NUM_CLASSES)
model = load_endovit_backbone(model, '/content/checkpoints/endovit_seg.pth')
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params/1e6:.1f}M — Trainable: {trainable/1e6:.1f}M')

In [ ]:
# ─── CELL 7: Train ───────────────────────────────────────────────
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

# ── Config ──
# Few-shot: use 4 training videos (1,9,12,17), validate on video18
TRAIN_VIDS = [1, 9, 12, 17]
VAL_VIDS   = [18]
BATCH_SIZE = 8
NUM_EPOCHS = 40
LR_BACKBONE = 1e-5   # lower LR for pretrained backbone
LR_HEAD      = 1e-4  # higher LR for new head

# ── Datasets ──
train_ds = CholecSeg8kDataset(DATA_ROOT, TRAIN_VIDS, IMAGE_SIZE, augment=True)
val_ds   = CholecSeg8kDataset(DATA_ROOT, VAL_VIDS,   IMAGE_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} samples | Val: {len(val_ds)} samples')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

# ── Optimizer with layer-wise LR ──
optimizer = optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': LR_BACKBONE, 'weight_decay': 0.05},
    {'params': list(model.reassemble.parameters()) +
               list(model.fusion.parameters()) +
               list(model.head.parameters()),
     'lr': LR_HEAD, 'weight_decay': 0.01},
])

scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

# ── Loss: CrossEntropy + Dice ──
ce_loss = nn.CrossEntropyLoss(ignore_index=255)

def dice_loss(pred, target, num_classes=13, eps=1e-6):
    pred_soft = torch.softmax(pred, dim=1)
    target_oh = torch.zeros_like(pred_soft)
    target_oh.scatter_(1, target.unsqueeze(1).clamp(0, num_classes-1), 1)
    intersection = (pred_soft * target_oh).sum(dim=(2,3))
    union = pred_soft.sum(dim=(2,3)) + target_oh.sum(dim=(2,3))
    dice = (2 * intersection + eps) / (union + eps)
    return 1 - dice.mean()

def combined_loss(pred, target):
    return ce_loss(pred, target) + 0.5 * dice_loss(pred, target)

# ── mIoU metric ──
def compute_miou(pred, target, num_classes=13):
    pred_cls = pred.argmax(dim=1)
    ious = []
    for cls in range(num_classes):
        p = (pred_cls == cls)
        t = (target == cls)
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        if union > 0:
            ious.append((inter / union).item())
    return np.mean(ious) if ious else 0.0

# ── Training loop ──
best_miou   = 0.0
best_ckpt   = '/content/checkpoints/surgivision_seg_best.pth'

print('\n🚀 Starting training...')
print(f'   Epochs: {NUM_EPOCHS} | Batch: {BATCH_SIZE} | Train vids: {TRAIN_VIDS}\n')

for epoch in range(NUM_EPOCHS):
    # ── Train ──
    model.train()
    train_loss = 0.0
    t0 = time.time()

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = combined_loss(preds, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    train_loss /= len(train_loader)

    # ── Validate every 5 epochs ──
    if (epoch + 1) % 5 == 0 or epoch == 0:
        model.eval()
        val_ious = []
        with torch.no_grad():
            for imgs, masks in val_loader:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                preds = model(imgs)
                val_ious.append(compute_miou(preds, masks))

        val_miou = np.mean(val_ious)
        elapsed  = time.time() - t0

        print(f'Epoch [{epoch+1:3d}/{NUM_EPOCHS}] '
              f'Loss: {train_loss:.4f} | '
              f'Val mIoU: {val_miou*100:.1f}% | '
              f'LR: {scheduler.get_last_lr()[1]:.2e} | '
              f'Time: {elapsed:.0f}s')

        # Save best
        if val_miou > best_miou:
            best_miou = val_miou
            torch.save({
                'epoch':      epoch + 1,
                'model':      model.state_dict(),
                'optimizer':  optimizer.state_dict(),
                'val_miou':   val_miou,
                'train_vids': TRAIN_VIDS,
                'num_classes': NUM_CLASSES,
                'image_size': IMAGE_SIZE,
            }, best_ckpt)
            print(f'  ✓ New best! mIoU: {best_miou*100:.1f}% saved to {best_ckpt}')
    else:
        elapsed = time.time() - t0
        print(f'Epoch [{epoch+1:3d}/{NUM_EPOCHS}] Loss: {train_loss:.4f} | Time: {elapsed:.0f}s')

print(f'\n🎉 Training complete! Best mIoU: {best_miou*100:.1f}%')
print(f'   Checkpoint: {best_ckpt}')

In [ ]:
# ─── CELL 8: Visualize predictions ───────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

CLASS_COLORS = [
    [10,  10,  10],   # 0  Background
    [192, 128,  96],  # 1  Abdominal Wall
    [196,  48,  32],  # 2  Liver
    [160,  96,  64],  # 3  Gastrointestinal
    [208, 176,  64],  # 4  Fat
    [192, 200, 208],  # 5  Grasper
    [176, 128, 128],  # 6  Connective Tissue
    [255,  32,  64],  # 7  Blood
    [224,  48,  80],  # 8  Cystic Duct
    [224, 224, 255],  # 9  L-Hook Electrocautery
    [ 64, 160,  64],  # 10 Gallbladder
    [128,  64, 192],  # 11 Hepatic Vein
    [128,  64,  32],  # 12 Liver Ligament
]
CLASS_NAMES = [
    'Background', 'Abdominal Wall', 'Liver', 'Gastrointestinal',
    'Fat', 'Grasper', 'Connective Tissue', 'Blood', 'Cystic Duct',
    'L-Hook Cautery', 'Gallbladder', 'Hepatic Vein', 'Liver Ligament'
]

def mask_to_rgb(mask_np):
    rgb = np.zeros((*mask_np.shape, 3), dtype=np.uint8)
    for cls_idx, color in enumerate(CLASS_COLORS):
        rgb[mask_np == cls_idx] = color
    return rgb

# Load best checkpoint
ckpt = torch.load(best_ckpt, map_location=DEVICE)
model.load_state_dict(ckpt['model'])
model.eval()

# Visualize 4 samples
fig, axes = plt.subplots(4, 3, figsize=(15, 20))
fig.suptitle(f'EndoViT Segmentation — Val mIoU: {ckpt["val_miou"]*100:.1f}%', fontsize=16)

denorm_mean = torch.tensor([0.3464, 0.2280, 0.2228]).view(3,1,1)
denorm_std  = torch.tensor([0.2520, 0.2128, 0.2093]).view(3,1,1)

val_iter = iter(val_loader)
imgs, masks = next(val_iter)

with torch.no_grad():
    preds = model(imgs.to(DEVICE)).cpu()

for i in range(min(4, len(imgs))):
    img_show  = (imgs[i] * denorm_std + denorm_mean).clamp(0,1).permute(1,2,0).numpy()
    gt_show   = mask_to_rgb(masks[i].numpy())
    pred_show = mask_to_rgb(preds[i].argmax(0).numpy())

    axes[i,0].imshow(img_show); axes[i,0].set_title('Input Frame'); axes[i,0].axis('off')
    axes[i,1].imshow(gt_show);  axes[i,1].set_title('Ground Truth'); axes[i,1].axis('off')
    axes[i,2].imshow(pred_show);axes[i,2].set_title('Prediction'); axes[i,2].axis('off')

plt.tight_layout()
plt.savefig('/content/checkpoints/predictions_preview.png', dpi=100, bbox_inches='tight')
plt.show()
print('✓ Visualization saved')

In [ ]:
# ─── CELL 9: Download checkpoint ─────────────────────────────────
# This downloads the trained model to your Mac.
# Place it at:
#   /Users/matteomeister/Documents/Medical Devices/Projects/EndoViT/
#   pretraining/pretrained_endovit_models/EndoViT_for_Segmentation/surgivision_seg_best.pth
# Then restart surgivision_server.py

from google.colab import files

print(f'Checkpoint info:')
ckpt_info = torch.load(best_ckpt, map_location='cpu')
print(f'  Epoch:      {ckpt_info["epoch"]}')
print(f'  Val mIoU:   {ckpt_info["val_miou"]*100:.1f}%')
print(f'  Train vids: {ckpt_info["train_vids"]}')
print(f'  Classes:    {ckpt_info["num_classes"]}')
print()
print('Downloading...')
files.download(best_ckpt)
files.download('/content/checkpoints/predictions_preview.png')
print('✓ Done! Place surgivision_seg_best.pth in:')
print('  EndoViT/pretraining/pretrained_endovit_models/EndoViT_for_Segmentation/')